# Task 3: Event Impact Modeling

## Forecasting Financial Inclusion in Ethiopia

This notebook models preliminary relationships between major financial-
inclusion events and Ethiopia's Access and Usage indicators.

The analysis:

- connects event records to impact-link records;
- creates an event-indicator association matrix;
- represents event effects using direction, magnitude, and lag;
- compares selected estimates with observed historical changes;
- documents assumptions, uncertainty, and limitations.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "ethiopia_fi_enriched.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR = PROJECT_ROOT / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found: {DATA_PATH}"
    )

print("Environment ready")

Project root: /Users/mac/Ethiopia-FI-Interim
Dataset path: /Users/mac/Ethiopia-FI-Interim/data/processed/ethiopia_fi_enriched.csv
Environment ready


In [2]:
df = pd.read_csv(DATA_PATH)

# Normalize column names.
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

# Normalize record types.
df["record_type_normalized"] = (
    df["record_type"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

# Convert numeric fields where available.
numeric_columns = [
    "value_numeric",
    "impact_magnitude",
    "lag_months",
    "confidence_score"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

# Convert possible date fields.
date_columns = [
    "observation_date",
    "event_date",
    "collection_date"
]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(
            df[column],
            errors="coerce"
        )

print("Dataset shape:", df.shape)

print("\nRecord types:")
print(
    df["record_type_normalized"]
    .value_counts()
)

print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (62, 36)

Record types:
record_type_normalized
observation    33
impact_link    15
event          11
target          3
Name: count, dtype: int64

Available columns:
['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes', 'parent_id', 'record_type_normalized']


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes,parent_id,record_type_normalized
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaN,NaN,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,Baseline year,NaN,NaN,observation
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaN,NaN,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,NaN,NaN,NaN,observation
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,NaN,NaN,NaN,observation
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,Gender disaggregated,NaN,NaN,observation
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaT,Gender disaggregated,NaN,NaN,observation


In [3]:
events = df[
    df["record_type_normalized"].eq("event")
].copy()

impact_links = df[
    df["record_type_normalized"].eq("impact_link")
].copy()

print("Event records:", len(events))
print("Impact-link records:", len(impact_links))

event_columns = [
    column for column in [
        "record_id",
        "indicator",
        "indicator_code",
        "category",
        "event_date",
        "observation_date",
        "source_name",
        "confidence"
    ]
    if column in events.columns
]

impact_columns = [
    column for column in [
        "record_id",
        "parent_id",
        "pillar",
        "related_indicator",
        "indicator_code",
        "impact_direction",
        "impact_magnitude",
        "lag_months",
        "evidence_basis",
        "source_name",
        "confidence"
    ]
    if column in impact_links.columns
]

print("\nEvents")
display(events[event_columns])

print("\nImpact links")
display(impact_links[impact_columns])

Event records: 11
Impact-link records: 15

Events


,record_id,indicator,indicator_code,category,observation_date,source_name,confidence
33,EVT_0001,Telebirr Launch,EVT_TELEBIRR,product_launch,2021-05-17,Ethio Telecom,high
34,EVT_0002,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,market_entry,2022-08-01,News,high
35,EVT_0003,M-Pesa Ethiopia Launch,EVT_MPESA,product_launch,2023-08-01,Safaricom,high
36,EVT_0004,Fayda Digital ID Program Rollout,EVT_FAYDA,infrastructure,2024-01-01,NIDP,high
37,EVT_0005,Foreign Exchange Liberalization,EVT_FX_REFORM,policy,2024-07-29,NBE,high
38,EVT_0006,P2P Transaction Count Surpasses ATM,EVT_CROSSOVER,milestone,2024-10-01,EthSwitch,high
39,EVT_0007,M-Pesa EthSwitch Integration,EVT_MPESA_INTEROP,partnership,2025-10-27,EthSwitch,high
40,EVT_0008,EthioPay Instant Payment System Launch,EVT_ETHIOPAY,infrastructure,2025-12-18,NBE/EthSwitch,high
41,EVT_0009,NFIS-II Strategy Launch,EVT_NFIS2,policy,2021-09-01,NBE,high
42,EVT_0010,Safaricom Ethiopia Price Increase,EVT_SAFCOM_PRICE,pricing,2025-12-15,News,high



Impact links


,record_id,parent_id,pillar,related_indicator,indicator_code,impact_direction,impact_magnitude,lag_months,evidence_basis,source_name,confidence
43,IMP_0001,EVT_0001,ACCESS,ACC_OWNERSHIP,NaN,increase,NaN,12.0,literature,NaN,medium
44,IMP_0002,EVT_0001,USAGE,USG_TELEBIRR_USERS,NaN,increase,NaN,3.0,empirical,NaN,high
45,IMP_0003,EVT_0001,USAGE,USG_P2P_COUNT,NaN,increase,NaN,6.0,empirical,NaN,medium
46,IMP_0004,EVT_0002,ACCESS,ACC_4G_COV,NaN,increase,NaN,12.0,empirical,NaN,medium
47,IMP_0005,EVT_0002,AFFORDABILITY,AFF_DATA_INCOME,NaN,decrease,NaN,12.0,literature,NaN,medium
48,IMP_0006,EVT_0003,USAGE,USG_MPESA_USERS,NaN,increase,NaN,3.0,empirical,NaN,high
49,IMP_0007,EVT_0003,ACCESS,ACC_MM_ACCOUNT,NaN,increase,NaN,6.0,theoretical,NaN,medium
50,IMP_0008,EVT_0004,ACCESS,ACC_OWNERSHIP,NaN,increase,NaN,24.0,literature,NaN,medium
51,IMP_0009,EVT_0004,GENDER,GEN_GAP_ACC,NaN,decrease,NaN,24.0,literature,NaN,medium
52,IMP_0010,EVT_0005,AFFORDABILITY,AFF_DATA_INCOME,NaN,increase,NaN,3.0,empirical,NaN,high


In [4]:
# Resolve the best available event-name column.
event_name_candidates = [
    "indicator",
    "event_name",
    "indicator_name",
    "description",
    "indicator_code",
    "record_id"
]

event_name_column = next(
    (
        column
        for column in event_name_candidates
        if column in events.columns
    ),
    None
)

if event_name_column is None:
    raise KeyError(
        "No event-name column could be identified."
    )

events["event_name_resolved"] = (
    events[event_name_column]
    .fillna("Unnamed event")
    .astype(str)
)

# Resolve event dates from any usable date column.
events["event_date_resolved"] = pd.NaT

for column in [
    "event_date",
    "observation_date",
    "date"
]:
    if column in events.columns:
        parsed = pd.to_datetime(
            events[column],
            errors="coerce"
        )

        events["event_date_resolved"] = (
            events["event_date_resolved"]
            .fillna(parsed)
        )

resolved_event_columns = [
    column for column in [
        "record_id",
        "event_name_resolved",
        "category",
        "event_date_resolved",
        "source_name",
        "confidence"
    ]
    if column in events.columns
]

events = events.sort_values(
    "event_date_resolved",
    na_position="last"
)

display(events[resolved_event_columns])

,record_id,event_name_resolved,category,event_date_resolved,source_name,confidence
33,EVT_0001,Telebirr Launch,product_launch,2021-05-17,Ethio Telecom,high
41,EVT_0009,NFIS-II Strategy Launch,policy,2021-09-01,NBE,high
34,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01,News,high
35,EVT_0003,M-Pesa Ethiopia Launch,product_launch,2023-08-01,Safaricom,high
36,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,2024-01-01,NIDP,high
37,EVT_0005,Foreign Exchange Liberalization,policy,2024-07-29,NBE,high
38,EVT_0006,P2P Transaction Count Surpasses ATM,milestone,2024-10-01,EthSwitch,high
39,EVT_0007,M-Pesa EthSwitch Integration,partnership,2025-10-27,EthSwitch,high
42,EVT_0010,Safaricom Ethiopia Price Increase,pricing,2025-12-15,News,high
40,EVT_0008,EthioPay Instant Payment System Launch,infrastructure,2025-12-18,NBE/EthSwitch,high


In [5]:
event_lookup_columns = [
    column for column in [
        "record_id",
        "event_name_resolved",
        "event_date_resolved",
        "category",
        "source_name",
        "confidence"
    ]
    if column in events.columns
]

event_lookup = events[
    event_lookup_columns
].copy()

event_lookup = event_lookup.rename(
    columns={
        "record_id": "event_record_id",
        "source_name": "event_source_name",
        "confidence": "event_confidence"
    }
)

impact_event_model = impact_links.merge(
    event_lookup,
    left_on="parent_id",
    right_on="event_record_id",
    how="left",
    validate="many_to_one"
)

print(
    "Impact links successfully matched to events:",
    impact_event_model["event_record_id"].notna().sum(),
    "of",
    len(impact_event_model)
)

unmatched = impact_event_model[
    impact_event_model["event_record_id"].isna()
]

if not unmatched.empty:
    print("\nUnmatched parent IDs:")
    display(
        unmatched[
            [
                column
                for column in [
                    "record_id",
                    "parent_id",
                    "related_indicator"
                ]
                if column in unmatched.columns
            ]
        ]
    )

summary_columns = [
    column for column in [
        "event_name_resolved",
        "event_date_resolved",
        "category",
        "pillar",
        "related_indicator",
        "impact_direction",
        "impact_magnitude",
        "lag_months",
        "evidence_basis",
        "confidence"
    ]
    if column in impact_event_model.columns
]

display(
    impact_event_model[summary_columns]
    .sort_values(
        [
            column
            for column in [
                "event_date_resolved",
                "related_indicator"
            ]
            if column in impact_event_model.columns
        ]
    )
)

Impact links successfully matched to events: 15 of 15


,event_name_resolved,event_date_resolved,pillar,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis,confidence
0,Telebirr Launch,2021-05-17,ACCESS,ACC_OWNERSHIP,increase,NaN,12.0,literature,medium
2,Telebirr Launch,2021-05-17,USAGE,USG_P2P_COUNT,increase,NaN,6.0,empirical,medium
1,Telebirr Launch,2021-05-17,USAGE,USG_TELEBIRR_USERS,increase,NaN,3.0,empirical,high
3,Safaricom Ethiopia Commercial Launch,2022-08-01,ACCESS,ACC_4G_COV,increase,NaN,12.0,empirical,medium
4,Safaricom Ethiopia Commercial Launch,2022-08-01,AFFORDABILITY,AFF_DATA_INCOME,decrease,NaN,12.0,literature,medium
6,M-Pesa Ethiopia Launch,2023-08-01,ACCESS,ACC_MM_ACCOUNT,increase,NaN,6.0,theoretical,medium
5,M-Pesa Ethiopia Launch,2023-08-01,USAGE,USG_MPESA_USERS,increase,NaN,3.0,empirical,high
7,Fayda Digital ID Program Rollout,2024-01-01,ACCESS,ACC_OWNERSHIP,increase,NaN,24.0,literature,medium
8,Fayda Digital ID Program Rollout,2024-01-01,GENDER,GEN_GAP_ACC,decrease,NaN,24.0,literature,medium
9,Foreign Exchange Liberalization,2024-07-29,AFFORDABILITY,AFF_DATA_INCOME,increase,NaN,3.0,empirical,high
